In [2]:
import pandas as pd
import json
import glob
import ijson


## Inspect user_id + labels dataset

In [3]:
df_split = pd.read_csv('./datasets/split.csv')
df_labels = pd.read_csv('./datasets/label.csv')
df_id_labels = df_split.merge(df_labels, on="id", how="inner")
df_id_labels

,id,split,label
0,u2664730894,train,human
1,u1089159225148882949,train,human
2,u36741729,train,bot
3,u1679822588,train,bot
4,u1519144464,train,human
...,...,...,...
999995,u1380005641863917569,test,human
999996,u815989128408006656,test,human
999997,u706703037973147652,test,human
999998,u238715475,test,human


## Sample 20k users, around 60:40 split between humans and bots

In [4]:
# split into humans and bot accounts first 
df_human = df_id_labels[df_id_labels["label"] == "human"]
df_bot = df_id_labels[df_id_labels["label"] == "bot"]

# apply random sampling in each label category dataset
# 61:39 split between humans and bots
sample_human = df_human.sample(n=12200, random_state=42)
sample_bot = df_bot.sample(n=7800, random_state=42)

df_20k = (
    pd.concat([sample_human, sample_bot], axis=0)
    # randomly shuffles rows to mix humans and bots
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
df_20k = df_20k.drop(columns=['split']) 

In [5]:
# check distribution between humans and bots 
df_20k.groupby(["label"]).size()

label
bot       7800
human    12200
dtype: int64

In [6]:
df_20k

,id,label
0,u1413937537421352972,human
1,u203565739,human
2,u425400861,human
3,u31731821,human
4,u1124500104071790593,bot
...,...,...
19995,u18599219,human
19996,u1472854555696279554,human
19997,u233345733,human
19998,u1468360896812847105,human


In [7]:
# export to csv
df_20k.to_csv("user_id_20k.csv", index=False)

## Merge with user metadata with user_id as primary key

In [8]:
import ijson
user_ids = set(df_20k["id"])
selected_ids = set(user_ids)
data = []

with open('./datasets/user.json', "rb") as f:
    users = ijson.items(f, "item")

    for user in users:
        if user["id"] in selected_ids:
            data.append(user)

df_user = pd.DataFrame(data)

In [9]:
# merging
df_user_20k = df_20k.merge(df_user, on="id", how="inner")
df_user_20k

,id,label,created_at,description,entities,location,name,pinned_tweet_id,profile_image_url,protected,public_metrics,url,username,verified,withheld
0,u1413937537421352972,human,2021-07-10 19:06:09+00:00,Wife_ step mom _ will be a GP _ cat person\nIn...,None,NaN,Lalibali,1.454441e+18,https://pbs.twimg.com/profile_images/141429174...,False,"{'followers_count': 692, 'following_count': 16...",,Lalibali12,False,None
1,u203565739,human,2010-10-16 15:32:38+00:00,A chapter of the American Academy of Pediatric...,"{'url': {'urls': [{'start': 0, 'end': 22, 'url...",Oregon,Oregon Pediatric Society,NaN,https://pbs.twimg.com/profile_images/117401664...,False,"{'followers_count': 1200, 'following_count': 2...",http://t.co/awjWTEyean,OregonAAP,False,None
2,u425400861,human,2011-11-30 23:32:32+00:00,El País México y América | SEO,None,NaN,Julieta Sanguino,NaN,https://pbs.twimg.com/profile_images/106651032...,False,"{'followers_count': 897, 'following_count': 39...",,Julaiilama,False,None
3,u31731821,human,2009-04-16 11:30:57+00:00,Artist,"{'url': {'urls': [{'start': 0, 'end': 23, 'url...","Seven Lakes, NC",Tess M Joseph,NaN,https://pbs.twimg.com/profile_images/108175547...,False,"{'followers_count': 73, 'following_count': 188...",https://t.co/gKx7qESWwI,VVMillinery,False,None
4,u1124500104071790593,bot,2019-05-04 02:24:51+00:00,,None,NaN,JK333,NaN,https://abs.twimg.com/sticky/default_profile_i...,False,"{'followers_count': 0, 'following_count': 80, ...",,animalvisceral,False,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,u18599219,human,2009-01-04 09:12:06+00:00,⠀ She/ Her \n#blogger #selfdiscoverycoach #cre...,"{'url': {'urls': [{'start': 0, 'end': 23, 'url...","Secunderabad, Hyderabad",Corinne Rodrigues,1.495586e+18,https://pbs.twimg.com/profile_images/129279551...,False,"{'followers_count': 4169, 'following_count': 2...",https://t.co/Te2ZDpGJik,CorinneBlogs,False,None
19996,u1472854555696279554,human,2021-12-20 09:01:18+00:00,Radio & Tv Host | Events Mc | Fashionista| \n=...,None,NaN,Mellon Trisha,NaN,https://pbs.twimg.com/profile_images/147285482...,False,"{'followers_count': 20, 'following_count': 46,...",,mellon_trisha,False,None
19997,u233345733,human,2011-01-03 00:31:11+00:00,I had a really great time last night,None,NaN,C.J. Garrett,1.267216e+18,https://pbs.twimg.com/profile_images/145524515...,False,"{'followers_count': 159, 'following_count': 18...",,GooseCJGarrett,False,None
19998,u1468360896812847105,human,2021-12-07 23:25:11+00:00,Sasha Banks is the greatest ever Nicki Minaj i...,None,NaN,Theresa Romano,NaN,https://pbs.twimg.com/profile_images/146836583...,False,"{'followers_count': 14, 'following_count': 4, ...",,TheresaRomano15,False,None


## break public metrics into individual columns

In [10]:
df_user_20k["public_metrics"]

0        {'followers_count': 692, 'following_count': 16...
1        {'followers_count': 1200, 'following_count': 2...
2        {'followers_count': 897, 'following_count': 39...
3        {'followers_count': 73, 'following_count': 188...
4        {'followers_count': 0, 'following_count': 80, ...
                               ...                        
19995    {'followers_count': 4169, 'following_count': 2...
19996    {'followers_count': 20, 'following_count': 46,...
19997    {'followers_count': 159, 'following_count': 18...
19998    {'followers_count': 14, 'following_count': 4, ...
19999    {'followers_count': 22, 'following_count': 634...
Name: public_metrics, Length: 20000, dtype: object

In [11]:
# break public metrics into individual columns
metrics_df = pd.json_normalize(df_user_20k["public_metrics"])
df_user_20k = pd.concat([df_user_20k.drop(columns=["public_metrics"]), metrics_df], axis=1)

In [12]:
df_user_20k

,id,label,created_at,description,entities,location,name,pinned_tweet_id,profile_image_url,protected,url,username,verified,withheld,followers_count,following_count,tweet_count,listed_count
0,u1413937537421352972,human,2021-07-10 19:06:09+00:00,Wife_ step mom _ will be a GP _ cat person\nIn...,None,NaN,Lalibali,1.454441e+18,https://pbs.twimg.com/profile_images/141429174...,False,,Lalibali12,False,None,692,1636,4169,0
1,u203565739,human,2010-10-16 15:32:38+00:00,A chapter of the American Academy of Pediatric...,"{'url': {'urls': [{'start': 0, 'end': 22, 'url...",Oregon,Oregon Pediatric Society,NaN,https://pbs.twimg.com/profile_images/117401664...,False,http://t.co/awjWTEyean,OregonAAP,False,None,1200,242,1602,24
2,u425400861,human,2011-11-30 23:32:32+00:00,El País México y América | SEO,None,NaN,Julieta Sanguino,NaN,https://pbs.twimg.com/profile_images/106651032...,False,,Julaiilama,False,None,897,394,360,6
3,u31731821,human,2009-04-16 11:30:57+00:00,Artist,"{'url': {'urls': [{'start': 0, 'end': 23, 'url...","Seven Lakes, NC",Tess M Joseph,NaN,https://pbs.twimg.com/profile_images/108175547...,False,https://t.co/gKx7qESWwI,VVMillinery,False,None,73,188,11352,0
4,u1124500104071790593,bot,2019-05-04 02:24:51+00:00,,None,NaN,JK333,NaN,https://abs.twimg.com/sticky/default_profile_i...,False,,animalvisceral,False,None,0,80,12,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,u18599219,human,2009-01-04 09:12:06+00:00,⠀ She/ Her \n#blogger #selfdiscoverycoach #cre...,"{'url': {'urls': [{'start': 0, 'end': 23, 'url...","Secunderabad, Hyderabad",Corinne Rodrigues,1.495586e+18,https://pbs.twimg.com/profile_images/129279551...,False,https://t.co/Te2ZDpGJik,CorinneBlogs,False,None,4169,2444,52367,220
19996,u1472854555696279554,human,2021-12-20 09:01:18+00:00,Radio & Tv Host | Events Mc | Fashionista| \n=...,None,NaN,Mellon Trisha,NaN,https://pbs.twimg.com/profile_images/147285482...,False,,mellon_trisha,False,None,20,46,3,0
19997,u233345733,human,2011-01-03 00:31:11+00:00,I had a really great time last night,None,NaN,C.J. Garrett,1.267216e+18,https://pbs.twimg.com/profile_images/145524515...,False,,GooseCJGarrett,False,None,159,185,11794,5
19998,u1468360896812847105,human,2021-12-07 23:25:11+00:00,Sasha Banks is the greatest ever Nicki Minaj i...,None,NaN,Theresa Romano,NaN,https://pbs.twimg.com/profile_images/146836583...,False,,TheresaRomano15,False,None,14,4,5705,0


In [13]:
df_user_20k = pd.read_csv('./datasets/user_metadata_20k.csv')

## Parse edges csv, filter for 20k users, output as dataframe/csv

### to discuss: following/follower edges extends to some users outside of the 20k sample

In [14]:
chunks = []
user_ids = set(df_user_20k["id"])
selected_ids = set(user_ids)
for chunk in pd.read_csv('./datasets/edge.csv', chunksize=1_000_000):
    filtered_chunk = chunk[
        chunk["source_id"].isin(selected_ids) |
        chunk["target_id"].isin(selected_ids)
    ]
    chunks.append(filtered_chunk)

df_edges_filtered = pd.concat(chunks, ignore_index=True)

## Keep user to user relations(following, follower) within the sample 20k users i.e: trim edges linking to outside users

In [15]:
# Your 20k user IDs with "u" prefix
user_20k_ids = set(df_user_20k["id"])  # already has "u" prefix

# Separate user-user edges and non-user-user edges
user_user_relations = {"following", "followers"}

df_edges_user_user = df_edges_filtered[
    df_edges_filtered["relation"].isin(user_user_relations)
]

df_edges_non_user_user = df_edges_filtered[
    ~df_edges_filtered["relation"].isin(user_user_relations)
]

# Filter user-user edges to strictly within 20k
df_edges_user_user_filtered = df_edges_user_user[
    df_edges_user_user["source_id"].isin(user_20k_ids) &
    df_edges_user_user["target_id"].isin(user_20k_ids)
]

print(f"Original user-user edges: {len(df_edges_user_user)}")
print(f"Within 20k user-user edges: {len(df_edges_user_user_filtered)}")

# Recombine
df_edges_final = pd.concat([df_edges_user_user_filtered, df_edges_non_user_user], ignore_index=True)
print(f"Final edges: {len(df_edges_final)}")

df_edges_final.to_csv('./datasets/edges_final.csv', index=False)
print("Done!")

Original user-user edges: 120914
Within 20k user-user edges: 989
Final edges: 1641930
Done!


In [16]:
df_edges_filtered = pd.read_csv('./datasets/edges_filtered_users.csv')
print(df_edges_filtered)

                    source_id    relation             target_id
0                   u15231287   following            u130620969
1                  u376044283   following            u236554385
2        u1233813128556765186   following             u33884545
3         u974697596085329921   followers            u133836828
4         u909451577999724544   followers            u540223123
...                       ...         ...                   ...
1761850  l1327101661937020928  membership             u18193572
1761851             l25817931  membership            u169407655
1761852             l95536191  membership           u1911644054
1761853            l234600495  membership  u1011387798518157312
1761854              l4625623  membership           u2645516899

[1761855 rows x 3 columns]


### theres alot more outsiders than 20k sample users

In [17]:

# Get all user IDs appearing in edges
all_user_ids_in_edges = {
    i for i in 
    set(df_edges_filtered["source_id"]) | set(df_edges_filtered["target_id"])
    if str(i).startswith("u")
}

# Your 20k user IDs with "u" prefix
user_20k_ids = set(df_user_20k["id"])  # already has "u" prefix

# Users in edges but outside your 20k
outsider_users = all_user_ids_in_edges - user_20k_ids

print(f"Total user nodes in edges: {len(all_user_ids_in_edges)}")
print(f"Your 20k users: {len(user_20k_ids)}")
print(f"Outsider users: {len(outsider_users)}")

# Sample some outsiders to verify
import random
print("\nSample outsider IDs:")
print(random.sample(list(outsider_users), 5))

Total user nodes in edges: 69124
Your 20k users: 20000
Outsider users: 49134

Sample outsider IDs:
['u3351632566', 'u237765532', 'u302152200', 'u720963865660506113', 'u1362800101639426052']


## Important step
## Second pass through edges csv to filter for all tweet ids in df_edges_filtered, instead of user ids
## capture rows in edges.csv with tweet-tweet, list-tweet, tweet-hashtag relations (indirect relation between users: user --> tweet ---> tweet --> user)

###  whether we include outsiders or not doesnt affect this process, since we re just filtering for tweet ids (only user ids are corrupted with outsiders and have to be trimmed)

In [18]:
# Already done
all_ids = set(df_edges_final["source_id"]).union(set(df_edges_final["target_id"]))
tweet_ids_pass1 = {i for i in all_ids if str(i).startswith("t")}
print(f"Pass 1 tweet IDs: {len(tweet_ids_pass1)}")

Pass 1 tweet IDs: 1602556


In [19]:
relations_found = set()

for i, chunk in enumerate(pd.read_csv('./datasets/edge.csv', 
                                       chunksize=10_000_000,
                                       usecols=["relation"])):  # only load relation column, much faster
    relations_found.update(chunk["relation"].unique())
    print(f"Chunk {i}: {relations_found}")

print(f"\nAll relations in file: {relations_found}")

Chunk 0: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 1: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 2: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 3: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 4: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 5: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 6: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 7: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 8: {'own', 'following', 'post', 'pinned', 'followers'}
Chunk 9: {'own', 'contain', 'discuss', 'following', 'post', 'pinned', 'followers'}
Chunk 10: {'own', 'contain', 'discuss', 'following', 'post', 'pinned', 'followers'}
Chunk 11: {'own', 'contain', 'discuss', 'following', 'post', 'pinned', 'followers'}
Chunk 12: {'own', 'contain', 'discuss', 'following', 'post', 'pinned', 'followers'}
Chunk 13: {'own', 'contain', 'discuss', 'following', 'post', 'pinned', 'followers'}
Chunk 14: {'own', 'contain', 'di

## Check optimal chunk size (ideally 500mb - 1 GB)

In [20]:
chunk = next(pd.read_csv('./datasets/edge.csv', chunksize=10_000_000))
print(f"1M rows = {chunk.memory_usage(deep=True).sum() / 1e6:.1f} MB")

1M rows = 602.0 MB


In [21]:
# looking out for specific relation value strings 
second_pass_relations = {"retweeted", "quoted", "replied_to", "contain", "discuss"}

output_path = './datasets/edges_pass2_filtered.csv'
first = True
total = 0

for i, chunk in enumerate(pd.read_csv('./datasets/edge.csv', chunksize=10_000_000)):
    if i < 9:  # skip chunks with only following/followers/post/pin
        continue
    
    filtered = chunk[
        (chunk["relation"].isin(second_pass_relations)) &
        (
            chunk["source_id"].isin(tweet_ids_pass1) |
            chunk["target_id"].isin(tweet_ids_pass1)
        )
    ]
    
    if not filtered.empty:
        filtered.to_csv(output_path,
                        mode='w' if first else 'a',
                        header=first,
                        index=False)
        first = False
        total += len(filtered)
    
    print(f"Chunk {i}: matched so far: {total}")

print(f"Done! {total} edges saved")

Chunk 9: matched so far: 153039
Chunk 10: matched so far: 367562
Chunk 11: matched so far: 582134
Chunk 12: matched so far: 798198
Chunk 13: matched so far: 1013605
Chunk 14: matched so far: 1229937
Chunk 15: matched so far: 1446223
Chunk 16: matched so far: 1558693
Chunk 17: matched so far: 1558693
Done! 1558693 edges saved


## Save indirect edge relatons as dataframe, merge with user-based edge df

In [22]:
df_edges_pass2 = pd.read_csv('./datasets/edges_pass2_filtered.csv')


## edge df with tweet-tweet, list-tweet, tweet-hashtag relations  

In [23]:
df_edges_pass2['relation'].unique()

<ArrowStringArray>
['contain', 'discuss', 'quoted', 'retweeted', 'replied_to']
Length: 5, dtype: str

## edge df with only user to ___ , or ___ to user relations

In [24]:
df_edges_filtered['relation'].unique()

<ArrowStringArray>
[ 'following',  'followers',        'own',     'pinned',       'post',
  'mentioned',       'like',   'followed', 'membership']
Length: 9, dtype: str

In [25]:
df_edges_combined = pd.concat([df_edges_final, df_edges_pass2], ignore_index=True)

print(f"Original edges: {len(df_edges_filtered)}")
print(f"Pass 2 edges: {len(df_edges_pass2)}")
print(f"Combined edges: {len(df_edges_combined)}")

df_edges_combined.to_csv('./datasets/edges_combined_filtered.csv', index=False)
print("Done!")

Original edges: 1761855
Pass 2 edges: 1558693
Combined edges: 3200623
Done!


In [26]:
df_edges_combined

,source_id,relation,target_id
0,u567961995,following,u28075840
1,u939975090434904069,following,u1270906326823186432
2,u58888683,following,u1066118097566920704
3,u996829570274807808,followers,u1425312010020003840
4,u13727002,followers,u1177236995447427072
...,...,...,...
3200618,t1461034869425946625,retweeted,t1460996439493029896
3200619,t1464325730603614209,replied_to,t1464324802307629065
3200620,t1485940157467353089,retweeted,t1485937862637932553
3200621,t1227340265125502978,retweeted,t1227313497681342469


## Using all unique tweet ids from edges_combined, filter all 9 tweet.json files and merge them into one csv

In [27]:
import ijson
import glob
import pandas as pd
from csv import DictWriter
import ijson.common

# Collect tweet IDs from edges
all_ids = set(df_edges_combined["source_id"]).union(set(df_edges_combined["target_id"]))
tweet_ids = {i for i in all_ids if str(i).startswith("t")}
print(f"Unique tweet IDs to filter for: {len(tweet_ids)}")

CHUNK_SIZE = 50_000

for tweet_file in sorted(glob.glob('./datasets/tweet_*.json')):
    out_file = tweet_file.replace(".json", "_filtered.csv")
    chunk = []
    writer = None
    matched = 0

    with open(tweet_file, "rb") as fin, \
         open(out_file, 'w', newline='', encoding='utf-8') as csv_file:

        try:
            for tweet in ijson.items(fin, "item", use_float=True):
                if tweet.get("id") not in tweet_ids:
                    continue

                chunk.append(tweet)
                matched += 1

                if writer is None:
                    fieldnames = list(tweet.keys())
                    writer = DictWriter(csv_file, fieldnames=fieldnames, extrasaction='ignore')
                    writer.writeheader()

                if len(chunk) >= CHUNK_SIZE:
                    writer.writerows(chunk)
                    chunk.clear()

        except ijson.common.IncompleteJSONError:
            print(f"  ⚠️ {tweet_file} is incomplete/truncated — saving {matched} matched so far")

        finally:
            if chunk and writer:
                writer.writerows(chunk)
                chunk.clear()

    print(f"{tweet_file}: {matched} matched → {out_file}")

print("Done!")

Unique tweet IDs to filter for: 1660290
Done!


In [28]:
df_edges_filtered['relation'].unique()

<ArrowStringArray>
[ 'following',  'followers',        'own',     'pinned',       'post',
  'mentioned',       'like',   'followed', 'membership']
Length: 9, dtype: str

retweeted, quoted, reply_to, contain, discuss

In [29]:
# merging all 8 filtered csvs into one 
import glob

files = sorted(glob.glob('./datasets/tweet_*_filtered.csv'))
print(f"Found {len(files)} files to merge")

merged_path = './datasets/tweets_filtered_all.csv'

first = True
for f in files:
    print(f"Reading {f}...")
    for chunk in pd.read_csv(f, chunksize=50_000):
        chunk.to_csv(merged_path, 
                     mode='w' if first else 'a',  # write first time, append after
                     header=first,                 # only write header once
                     index=False)
        first = False

print(f"Done! Merged to {merged_path}")

Found 0 files to merge
Done! Merged to ./datasets/tweets_filtered_all.csv


In [30]:
df_tweets_all = pd.read_csv('./datasets/tweets_filtered_all.csv')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17628\1875295274.py:1: DtypeWarning: Columns (0: attachments, 1: context_annotations, 2: referenced_tweets, 3: reply_settings, 4: withheld) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tweets_all = pd.read_csv('./datasets/tweets_filtered_all.csv')


In [31]:
df_tweets_all.head()

,attachments,author_id,context_annotations,conversation_id,created_at,entities,geo,id,in_reply_to_user_id,lang,possibly_sensitive,public_metrics,referenced_tweets,reply_settings,source,text,withheld
0,NaN,1181610909053140993,NaN,1498104431744884745,2022-02-28 01:15:04+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1498104431744884745,NaN,en,False,"{'retweet_count': 2, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",One more day to apply! https://t.co/qEm0qQJcQs,NaN
1,NaN,1181610909053140993,NaN,1490723753222975496,2022-02-07 16:26:53+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1490723753222975496,8.604002e+08,en,False,"{'retweet_count': 0, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",@h_i_g_s_c_h @derspiegel This is so cool! Such...,NaN
2,NaN,1181610909053140993,NaN,1486427676583895044,2022-01-26 19:55:49+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1486427676583895044,1.306795e+18,en,False,"{'retweet_count': 0, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",@hanley_hans This is so neat! Really cool meth...,NaN
3,NaN,1181610909053140993,NaN,1485053368867381258,2022-01-23 00:54:48+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1485053368867381258,NaN,en,False,"{'retweet_count': 2, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",RT @gobIinbee: absolutely brilliant (intellige...,NaN
4,NaN,1181610909053140993,NaN,1483564412569071619,2022-01-18 22:18:13+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1483564412569071619,NaN,en,False,"{'retweet_count': 22, 'reply_count': None, 'li...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",RT @WinWithoutWar: 📅 EVENT REMINDER 📅\n\nJoin ...,NaN


In [ ]:
# Save as parquet after merging
df_tweets_all.to_parquet('./datasets/tweets_filtered_all.parquet')

# Future reads are ~5-10x faster
df_test = pd.read_parquet('./datasets/tweets_filtered_all.parquet')

In [33]:
df_test.head()

,attachments,author_id,context_annotations,conversation_id,created_at,entities,geo,id,in_reply_to_user_id,lang,possibly_sensitive,public_metrics,referenced_tweets,reply_settings,source,text,withheld
0,NaN,1181610909053140993,NaN,1498104431744884745,2022-02-28 01:15:04+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1498104431744884745,NaN,en,False,"{'retweet_count': 2, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",One more day to apply! https://t.co/qEm0qQJcQs,NaN
1,NaN,1181610909053140993,NaN,1490723753222975496,2022-02-07 16:26:53+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1490723753222975496,8.604002e+08,en,False,"{'retweet_count': 0, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",@h_i_g_s_c_h @derspiegel This is so cool! Such...,NaN
2,NaN,1181610909053140993,NaN,1486427676583895044,2022-01-26 19:55:49+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1486427676583895044,1.306795e+18,en,False,"{'retweet_count': 0, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",@hanley_hans This is so neat! Really cool meth...,NaN
3,NaN,1181610909053140993,NaN,1485053368867381258,2022-01-23 00:54:48+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1485053368867381258,NaN,en,False,"{'retweet_count': 2, 'reply_count': None, 'lik...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",RT @gobIinbee: absolutely brilliant (intellige...,NaN
4,NaN,1181610909053140993,NaN,1483564412569071619,2022-01-18 22:18:13+00:00,"{'hashtags': [], 'symbols': [], 'user_mentions...",NaN,t1483564412569071619,NaN,en,False,"{'retweet_count': 22, 'reply_count': None, 'li...",NaN,NaN,"<a href=""https://mobile.twitter.com"" rel=""nofo...",RT @WinWithoutWar: 📅 EVENT REMINDER 📅\n\nJoin ...,NaN


In [34]:
# Check for duplicates on tweet id
print(f"Total rows: {len(df_test)}")
print(f"Unique tweet IDs: {df_test['id'].nunique()}")
print(f"Duplicate rows: {df_test.duplicated(subset='id').sum()}")

Total rows: 1660290
Unique tweet IDs: 1660290
Duplicate rows: 0


## Sample of what tweet.json raw file looks like

{
  "id": "t1497798545872588801",
  "author_id": 1304855289208819713,
  "conversation_id": 1497798545872588801,
  "created_at": "2022-02-27 04:59:35+00:00",
  "lang": "en",
  "text": "@phaseknight_ Although I didn't base this sketch on you specifically, I really think it vibes with your amazing NSFW Laudna cosplay! 🖤 https://t.co/GXjjq83Rrn",
  "source": "Twitter for Android",
  "in_reply_to_user_id": 976935699793539073,
  "possibly_sensitive": false,
  "reply_settings": null,
  "withheld": null,
  "geo": null,
  "attachments": null,
  "context_annotations": null,
  "referenced_tweets": null,
  "public_metrics": {
    "retweet_count": 0,
    "reply_count": null,
    "like_count": 8,
    "quote_count": null
  },
  "entities": {
    "hashtags": [],
    "symbols": [],
    "urls": [],
    "user_mentions": [
      {
        "id": 976935699793539073,
        "id_str": "976935699793539073",
        "screen_name": "phaseknight_",
        "name": "sapphire starlight 💙✨",
        "indices": [0, 13]
      }
    ],
    "media": [
      {
        "id": 1497798542869422086,
        "id_str": "1497798542869422086",
        "type": "photo",
        "url": "https://t.co/GXjjq83Rrn",
        "display_url": "pic.twitter.com/GXjjq83Rrn",
        "expanded_url": "https://twitter.com/cbtillustrates/status/1497798545872588801/photo/1",
        "media_url": "http://pbs.twimg.com/media/FMk_1szUYAYeJ7N.jpg",
        "media_url_https": "https://pbs.twimg.com/media/FMk_1szUYAYeJ7N.jpg",
        "indices": [135, 158],
        "sizes": {
          "large":  {"w": 1575, "h": 2025, "resize": "fit"},
          "medium": {"w": 933,  "h": 1200, "resize": "fit"},
          "small":  {"w": 529,  "h": 680,  "resize": "fit"},
          "thumb":  {"w": 150,  "h": 150,  "resize": "crop"}
        }
      }
    ]
  }
}

## Convert list.json to list.csv

In [36]:
df_list = pd.read_json('./datasets/list.json')
print(df_list.shape)
print(df_list.head())

df_list.to_csv('./datasets/list.csv', index=False)
print("Done!")

(21870, 8)
                    id                      name                created_at  \
0             l1128774  StimulatingBroadband.com 2009-10-30 14:38:00+00:00   
1  l733341248057180166                Local news 2016-05-19 16:59:13+00:00   
2             l7983816                 Futerrans 2010-02-26 18:01:03+00:00   
3             l7008265          Beautiful People 2010-02-07 13:40:48+00:00   
4             l6807414               celebrities 2010-02-03 09:56:17+00:00   

                                         description  follower_count  \
0  Subscribe to our Twitter firehose of 500 telec...              35   
1  Follow for breaking news and updates in North ...               3   
2                     Futerra staff past and present               6   
3  People who are strong minded and stand up for ...              28   
4                                                NaN               7   

   member_count  private   owner_id  
0           452    False   32745876  
1          

In [37]:
## inspect list dataframe
df_list

,id,name,created_at,description,follower_count,member_count,private,owner_id
0,l1128774,StimulatingBroadband.com,2009-10-30 14:38:00+00:00,Subscribe to our Twitter firehose of 500 telec...,35,452,False,32745876
1,l733341248057180166,Local news,2016-05-19 16:59:13+00:00,Follow for breaking news and updates in North ...,3,47,False,316508410
2,l7983816,Futerrans,2010-02-26 18:01:03+00:00,Futerra staff past and present,6,25,False,23102054
3,l7008265,Beautiful People,2010-02-07 13:40:48+00:00,People who are strong minded and stand up for ...,28,179,False,23246523
4,l6807414,celebrities,2010-02-03 09:56:17+00:00,NaN,7,6,False,16454856
...,...,...,...,...,...,...,...,...
21865,l863566424375209984,Deep learning,2017-05-14 01:27:34+00:00,NaN,0,0,False,107770718
21866,l10667576,divulgacion & educacion,2010-04-16 10:03:29+00:00,NaN,29,373,False,29024460
21867,l90172505,Post Graphics Staff,2013-05-23 15:20:46+00:00,The visual journalists who work in The Washing...,67,32,False,87968068
21868,l702528800886648832,General science,2016-02-24 16:21:33+00:00,NaN,0,4,False,19732234


In [ ]:
import pandas as pd
import numpy as np

# Get unique nodes per type
user_ids = user_20k_ids
tweet_ids_set = {i for i in set(df_edges_combined["source_id"]) | set(df_edges_combined["target_id"]) if str(i).startswith("t")}
hashtag_ids = {i for i in set(df_edges_combined["source_id"]) | set(df_edges_combined["target_id"]) if str(i).startswith("h")}
list_ids = {i for i in set(df_edges_combined["source_id"]) | set(df_edges_combined["target_id"]) if str(i).startswith("l")}

num_users = len(user_ids)
num_tweets = len(tweet_ids_set)
num_hashtags = len(hashtag_ids)
num_lists = len(list_ids)

print(f"Users: {num_users}, Tweets: {num_tweets}, Hashtags: {num_hashtags}, Lists: {num_lists}")

# Max possible edges per relation type
max_edges = {
    "following":   num_users * num_users,
    "followers":   num_users * num_users,
    "post":        num_users * num_tweets,
    "pinned":      num_users * num_tweets,
    "like":        num_users * num_tweets,
    "mentioned":   num_tweets * num_users,
    "retweeted":   num_tweets * num_tweets,
    "quoted":      num_tweets * num_tweets,
    "replied_to":  num_tweets * num_tweets,
    "own":         num_users * num_lists,
    "membership":  num_lists * num_users,
    "followed":    num_lists * num_users,
    "contain":     num_lists * num_tweets,
    "discuss":     num_tweets * num_hashtags,
}

# Actual edges per relation
actual_edges = df_edges_combined.groupby("relation").size().to_dict()

# Print sparsity per relation
print(f"\n{'Relation':<15} {'Actual':>10} {'Max Possible':>15} {'Density':>10} {'Sparsity':>10}")
print("-" * 65)
for rel, max_e in max_edges.items():
    actual = actual_edges.get(rel, 0)
    density = actual / max_e if max_e > 0 else 0
    sparsity = 1 - density
    print(f"{rel:<15} {actual:>10,} {max_e:>15,} {density:>10.6f} {sparsity:>10.6f}")

Users: 20000, Tweets: 1660290, Hashtags: 278215, Lists: 11771

Relation            Actual    Max Possible    Density   Sparsity
-----------------------------------------------------------------
following              668     400,000,000   0.000002   0.999998
followers              321     400,000,000   0.000001   0.999999
post             1,526,311  33,205,800,000   0.000046   0.999954
pinned               5,952  33,205,800,000   0.000000   1.000000
like                10,067  33,205,800,000   0.000000   1.000000
mentioned           71,691  33,205,800,000   0.000002   0.999998
retweeted           55,943 2,756,562,884,100   0.000000   1.000000
quoted               9,410 2,756,562,884,100   0.000000   1.000000
replied_to          39,966 2,756,562,884,100   0.000000   1.000000
own                    385     235,420,000   0.000002   0.999998
membership          16,159     235,420,000   0.000069   0.999931
followed            10,376     235,420,000   0.000044   0.999956
contain             

In [ ]:
# Only look at user-user relations
user_user_relations = {"following", "followers", "mentioned"}  # mentioned: tweet->user but connects users indirectly

# Direct user-user edges
df_uu = df_edges_combined[df_edges_combined["relation"].isin({"following", "followers"})]

# Unique user pairs
unique_pairs = df_uu[["source_id", "target_id"]].drop_duplicates()
print(f"Unique user-user pairs: {len(unique_pairs)}")

# Max possible pairs (directed graph)
max_pairs = num_users * (num_users - 1)
print(f"Max possible directed pairs: {max_pairs:,}")

density = len(unique_pairs) / max_pairs
print(f"User-user graph density: {density:.8f}")
print(f"Average connections per user: {len(df_uu) / num_users:.4f}")

# Check how many users have at least one connection
connected_users = set(df_uu["source_id"]) | set(df_uu["target_id"])
isolated_users = user_20k_ids - connected_users
print(f"\nUsers with at least one follow edge: {len(connected_users)}")
print(f"Isolated users (no follow edges): {len(isolated_users)}")
print(f"Isolated user ratio: {len(isolated_users)/num_users:.2%}")

Unique user-user pairs: 842
Max possible directed pairs: 399,980,000
User-user graph density: 0.00000211
Average connections per user: 0.0495

Users with at least one follow edge: 857
Isolated users (no follow edges): 19143
Isolated user ratio: 95.71%


In [ ]:
import networkx as nx

# Build full graph with all entity types
G = nx.Graph()  # undirected to check reachability

# Add all edges from df_edges
# Convert to list of tuples first
edges = list(zip(df_edges_combined["source_id"], df_edges_combined["target_id"]))
G.add_edges_from(edges)

print(f"Total nodes in graph: {G.number_of_nodes()}")
print(f"Total edges in graph: {G.number_of_edges()}")

# Check connected components
components = list(nx.connected_components(G))
print(f"\nNumber of connected components: {len(components)}")
print(f"Largest component size: {max(len(c) for c in components)}")
print(f"Smallest component size: {min(len(c) for c in components)}")

# How many of your 20k users are in the largest component
largest_component = max(components, key=len)
users_in_largest = largest_component & user_20k_ids
print(f"\n20k users in largest component: {len(users_in_largest)}")
print(f"20k users isolated: {num_users - len(users_in_largest)}")

# Average path length (expensive, sample instead)
sample_users = random.sample(list(users_in_largest), min(100, len(users_in_largest)))
subG = G.subgraph(sample_users)
if nx.is_connected(subG):
    print(f"\nAvg path length (sampled): {nx.average_shortest_path_length(subG):.2f}")

Total nodes in graph: 1969155
Total edges in graph: 3189915

Number of connected components: 2512
Largest component size: 1857180
Smallest component size: 2

20k users in largest component: 16333
20k users isolated: 3667


In [ ]:
# Find isolated users
isolated_users = user_20k_ids - largest_component

# Check if they have any edges at all
isolated_in_edges = df_edges_final[
    df_edges_final["source_id"].isin(isolated_users) |
    df_edges_final["target_id"].isin(isolated_users)
]
print(f"Isolated users with any edges: {isolated_in_edges['source_id'].nunique()}")
print(f"Relations for isolated users:\n{isolated_in_edges['relation'].value_counts()}")

# Check if truly isolated (no edges at all)
truly_isolated = isolated_users - set(isolated_in_edges["source_id"]) - set(isolated_in_edges["target_id"])
print(f"Truly isolated (zero edges): {len(truly_isolated)}")

Isolated users with any edges: 6038
Relations for isolated users:
relation
post          103043
mentioned       3275
pinned           528
followed         294
membership       191
own               33
like               4
followers          1
Name: count, dtype: int64
Truly isolated (zero edges): 1121


# Build Tweets DataFrame for sample users only

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# Load files
users = pd.read_csv("./datasets/user_metadata_20k.csv")
edges = pd.read_csv("./datasets/edges_filtered_users.csv")
tweets = pq.read_table(
    "./datasets/tweets_filtered_all.parquet",
    columns=["id", "author_id", "text", "created_at"]
).to_pandas()

# Normalize user IDs for direct comparison
users["user_id_norm"] = users["id"].astype(str).str.replace(r"^[A-Za-z]", "", regex=True)
tweets["author_id_str"] = tweets["author_id"].astype(str)

# Direct join: tweet author_id matches user id without the 'u'
tweets_users = tweets.merge(
    users[["id", "user_id_norm"]],
    left_on="author_id_str",
    right_on="user_id_norm",
    how="inner"
).rename(columns={"id_x": "tweet_id", "id_y": "user_id"})

# Keep useful columns
tweets_users = tweets_users[["user_id", "tweet_id", "created_at", "text"]]

# Group tweets under each user
user_tweets_df = (
    tweets_users
    .groupby("user_id")
    .agg({
        "tweet_id": list,
        "created_at": list,
        "text": list
    })
    .reset_index()
)

display(user_tweets_df)

## missing nodes are probably tweetless, with pure user - user relations since they are not isolated (but need to recheck logic for isolated analysis)

,user_id,tweet_id,created_at,text
0,u1000038752813797377,"[t1490761242734174215, t1490033888130912269, t...","[2022-02-07 18:55:51+00:00, 2022-02-05 18:45:3...","[Allah shi kyauta https://t.co/I4ucbEbLmb, Had..."
1,u1000116915082231809,"[t1489190870356459522, t1441719337694347267, t...","[2022-02-03 10:55:45+00:00, 2021-09-25 11:00:5...",[Check out my NFT listing on OpenSea! https://...
2,u1000128578644795392,"[t1290204808926781440, t1286687477047189504, t...","[2020-08-03 08:36:11+00:00, 2020-07-24 15:39:3...",[RT @todor_fintech: Forbes: 3 Important Ways A...
3,u100030567,"[t1500243056410472449, t1500241723414523909, t...","[2022-03-05 22:53:12+00:00, 2022-03-05 22:47:5...","[@langolomaximo Variado, no podría responderte..."
4,u100032683,"[t1502930572024426501, t1502930530546892801, t...","[2022-03-13 08:52:26+00:00, 2022-03-13 08:52:1...",[RT @jk_rowling: Absolutely spot on about the ...
...,...,...,...,...
18073,u998927994411454464,"[t1502935551053377538, t1502928778460336129, t...","[2022-03-13 09:12:13+00:00, 2022-03-13 08:45:1...",[RT @mikeconsultants: Various Vacancies \n\nRe...
18074,u99895754,"[t1495964083237318660, t1495964036151787523, t...","[2022-02-22 03:30:05+00:00, 2022-02-22 03:29:5...",[RT @SALASARBALAJIP1: श्री #बालाजी #दर्शन साला...
18075,u999355654182711298,"[t1491834930611040265, t1491494547540566021, t...","[2022-02-10 18:02:19+00:00, 2022-02-09 19:29:4...",[These healthy habits are key even after you'v...
18076,u999407489392181254,"[t1499663804745015298, t1499658977994551297, t...","[2022-03-04 08:31:28+00:00, 2022-03-04 08:12:1...",[@Orhanay70550696 Çok pardon da; vefat twitini...
